In [18]:
!pip install catboost

import pandas as pd
import numpy as np

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Metrics & splits
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score

# Pipelines & preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

# Gradient boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Hyperparameter tuning
import optuna

# Persistence
import joblib

RANDOM_STATE = 42
N_TRIALS = 50

In [11]:
df = pd.read_csv("/content/Dataset.csv")

df.columns = df.columns.str.strip()

df['Class'] = df['Class'].str.lower().str.strip()
df['Class_Binary'] = (df['Class'] == 'good').astype(int)

df.rename(columns={
    'Temp': 'temperature',
    'Humid (%)': 'humidity',
    'CO2 (pmm)': 'co2',
    'Light (Fux)': 'light'
}, inplace=True)

FEATURES = ['temperature', 'humidity', 'co2']
TARGET = 'Class_Binary'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (8246, 3) Test: (2749, 3)


In [10]:
df.head()

,Fruit,temperature,humidity,light,CO2 (pmm),Class,Class_Binary
0,Orange,22,95,7.358649,361,good,1
1,Orange,24,95,14.835898,370,bad,0
2,Pineapple,25,95,10.104045,355,bad,0
3,Banana,25,89,20.179643,388,good,1
4,Tomato,23,90,12.621448,316,good,1


In [14]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, FEATURES)
    ],
    remainder="drop"
)


In [15]:
pipelines = {
    "random_forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "xgboost": Pipeline([
        ("preprocess", preprocessor),
        ("model", XGBClassifier(
            eval_metric="logloss",
            random_state=RANDOM_STATE
        ))
    ]),

    "lightgbm": Pipeline([
        ("preprocess", preprocessor),
        ("model", LGBMClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "catboost": Pipeline([
        ("preprocess", preprocessor),
        ("model", CatBoostClassifier(
            loss_function="Logloss",
            verbose=0,
            random_state=RANDOM_STATE
        ))
    ]),

    "logistic": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=500,
            random_state=RANDOM_STATE
        ))
    ])
}


In [19]:
def optuna_objective(trial, model_name):

    original_pipeline = pipelines[model_name]
    pipeline = clone(original_pipeline) # Clone the pipeline for each trial

    if model_name == "random_forest":
        pipeline.set_params(
            model__n_estimators=trial.suggest_int("n_estimators", 100, 500),
            model__max_depth=trial.suggest_int("max_depth", 3, 30),
            model__min_samples_split=trial.suggest_int("min_samples_split", 2, 20)
        )

    elif model_name == "logistic":
        pipeline.set_params(
            model__C=trial.suggest_float("C", 0.01, 10.0, log=True)
        )

    elif model_name == "xgboost":
        pipeline.set_params(
            model__n_estimators=trial.suggest_int("n_estimators", 100, 500),
            model__max_depth=trial.suggest_int("max_depth", 3, 10),
            model__learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3)
        )

    elif model_name == "lightgbm":
        pipeline.set_params(
            model__n_estimators=trial.suggest_int("n_estimators", 100, 500),
            model__num_leaves=trial.suggest_int("num_leaves", 16, 128),
            model__learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3)
        )

    elif model_name == "catboost":
        pipeline.set_params(
            model__iterations=trial.suggest_int("iterations", 200, 600),
            model__depth=trial.suggest_int("depth", 4, 10),
            model__learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3)
        )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for tr_idx, val_idx in cv.split(X_train, y_train):
        pipeline.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        preds = pipeline.predict(X_train.iloc[val_idx])
        scores.append(f1_score(y_train.iloc[val_idx], preds))

    return np.mean(scores)

In [20]:
results = {}
studies = {}

for model_name in pipelines.keys():
    print(f"\n Optimizing {model_name}")
    study = optuna.create_study(direction="maximize")
    study.optimize(lambda t: optuna_objective(t, model_name),
                   n_trials=N_TRIALS,
                   show_progress_bar=True)

    results[model_name] = study.best_value
    studies[model_name] = study

    print(f"Best F1 ({model_name}): {study.best_value:.4f}")

[I 2026-01-08 14:58:51,599] A new study created in memory with name: no-name-d64c87ae-e866-4e45-b5e5-f47c44511242



 Optimizing random_forest


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-01-08 14:58:57,434] Trial 0 finished with value: 0.9346941296655322 and parameters: {'n_estimators': 230, 'max_depth': 9, 'min_samples_split': 7}. Best is trial 0 with value: 0.9346941296655322.
[I 2026-01-08 14:59:01,435] Trial 1 finished with value: 0.9327452526322026 and parameters: {'n_estimators': 176, 'max_depth': 19, 'min_samples_split': 16}. Best is trial 0 with value: 0.9346941296655322.
[I 2026-01-08 14:59:08,303] Trial 2 finished with value: 0.931063004771598 and parameters: {'n_estimators': 289, 'max_depth': 30, 'min_samples_split': 7}. Best is trial 0 with value: 0.9346941296655322.
[I 2026-01-08 14:59:17,215] Trial 3 finished with value: 0.9315082389287956 and parameters: {'n_estimators': 372, 'max_depth': 28, 'min_samples_split': 9}. Best is trial 0 with value: 0.9346941296655322.
[I 2026-01-08 14:59:26,404] Trial 4 finished with value: 0.9318187919991932 and parameters: {'n_estimators': 378, 'max_depth': 13, 'min_samples_split': 6}. Best is trial 0 with value: 0

[I 2026-01-08 15:03:41,576] A new study created in memory with name: no-name-a87df523-65ec-413c-b8ba-a3aaf8b9b855


...

[I 2026-01-08 15:06:02,991] Trial 0 finished with value: 0.9359735339397472 and parameters: {'iterations': 464, 'depth': 8, 'learning_rate': 0.059146444918327594}. Best is trial 0 with value: 0.9359735339397472.
[I 2026-01-08 15:06:08,131] Trial 1 finished with value: 0.9339366914692906 and parameters: {'iterations': 328, 'depth': 7, 'learning_rate': 0.2505761708625985}. Best is trial 0 with value: 0.9359735339397472.
[I 2026-01-08 15:06:14,609] Trial 2 finished with value: 0.9348686798411079 and parameters: {'iterations': 415, 'depth': 5, 'learning_rate': 0.01877826081736819}. Best is trial 0 with value: 0.9359735339397472.
[I 2026-01-08 15:06:35,101] Trial 3 finished with value: 0.9367949415726974 and parameters: {'iterations': 538, 'depth': 10, 'learning_rate': 0.026282510210612536}. Best is trial 3 with value: 0.9367949415726974.
[I 2026-01-08 15:06:40,355] Trial 4 finished with value: 0.9361553549196506 and parameters: {'iterations': 203, 'depth': 8, 'learning_rate': 0.03078568329

[I 2026-01-08 15:14:31,479] A new study created in memory with name: no-name-3549a342-154a-4063-be57-04a2125ec914


[I 2026-01-08 15:14:31,472] Trial 49 finished with value: 0.9344773421921128 and parameters: {'iterations': 386, 'depth': 6, 'learning_rate': 0.09711808487166305}. Best is trial 19 with value: 0.9373445257746565.
Best F1 (catboost): 0.9373

 Optimizing logistic


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-01-08 15:14:31,899] Trial 0 finished with value: 0.8459830288010286 and parameters: {'C': 3.3020752180393154}. Best is trial 0 with value: 0.8459830288010286.
[I 2026-01-08 15:14:32,169] Trial 1 finished with value: 0.8345031180436623 and parameters: {'C': 0.05060178417596759}. Best is trial 0 with value: 0.8459830288010286.
[I 2026-01-08 15:14:32,432] Trial 2 finished with value: 0.8345031180436623 and parameters: {'C': 0.0807630783150961}. Best is trial 0 with value: 0.8459830288010286.
[I 2026-01-08 15:14:32,753] Trial 3 finished with value: 0.8270680054731641 and parameters: {'C': 0.021089797790050588}. Best is trial 0 with value: 0.8459830288010286.
[I 2026-01-08 15:14:33,203] Trial 4 finished with value: 0.827382541642916 and parameters: {'C': 0.02336652396932702}. Best is trial 0 with value: 0.8459830288010286.
[I 2026-01-08 15:14:33,717] Trial 5 finished with value: 0.8409761598317564 and parameters: {'C': 0.48625871009825483}. Best is trial 0 with value: 0.845983028801

In [25]:
best_model_name = max(results, key=results.get)
print("\n Best model:", best_model_name)

best_pipeline = clone(pipelines[best_model_name]) # Get a fresh, unfitted clone
best_pipeline.set_params(**{
    f"model__{k}": v for k, v in studies[best_model_name].best_params.items()
})

best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1:", f1_score(y_test, y_pred))

joblib.dump(best_pipeline, "storage_quality_pipeline.pkl")



 Best model: catboost
Test Accuracy: 0.9428883230265551
Test F1: 0.9448154657293497


['storage_quality_pipeline.pkl']